# Stock Return Prediction using Triple-Barrier Labeling

## 0. Imports

In [2]:
import os
import numpy as np
from typing import Optional, Tuple, List, Dict
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

## 1. Label Construction

### Configuration

In [3]:
# --------------------------------------------------
# Config
# --------------------------------------------------
DATA_DIR = "./data"
PANEL_FILE = os.path.join(DATA_DIR, "vn30_panel_daily.csv")
PROCESSED_OUT = os.path.join(DATA_DIR, "processed_vn30_panel_daily.csv")

START = "2018-01-01"
END   = "2026-03-15"

VOL_LOOKBACK = 20          # rolling vol window

### Load raw panel data

In [4]:
# --------------------------------------------------
# Load panel
# --------------------------------------------------
df = pd.read_csv(PANEL_FILE, parse_dates=["date"])

# --------------------------------------------------
# Basic cleaning
# --------------------------------------------------
print("Before cleaning:", len(df))

# remove duplicates
df = df.drop_duplicates(subset=["symbol", "date"]).copy()

# remove impossible rows
df = df[
    (df["close"] > 0) &
    (df["open"] > 0) &
    (df["high"] > 0) &
    (df["low"] > 0) &
    (df["volume"] >= 0)
].copy()

df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

print("After cleaning:", len(df))

Before cleaning: 61490
After cleaning: 61489


### Compute Returns & Volatility

In [5]:
# --------------------------------------------------
# Returns and rolling volatility
# --------------------------------------------------
def add_returns_and_volatility(data: pd.DataFrame, vol_lookback: int = 20) -> pd.DataFrame:
    out = data.copy()

    # log return
    out["log_ret"] = (
        out.groupby("symbol")["close"]
        .transform(lambda s: np.log(s).diff())
    )

    # rolling realized volatility of daily log returns
    out["daily_vol"] = (
        out.groupby("symbol")["log_ret"]
        .transform(lambda s: s.rolling(vol_lookback, min_periods=vol_lookback).std())
    )

    return out

df = add_returns_and_volatility(df, vol_lookback=VOL_LOOKBACK)
print(df.head(5))

        date symbol  open  high   low  close   volume   log_ret  daily_vol
0 2018-07-02    ACB  6.87  6.87  6.33   6.43  6858520       NaN        NaN
1 2018-07-03    ACB  6.45  6.56  5.91   5.98  7375018 -0.072554        NaN
2 2018-07-04    ACB  5.89  6.18  5.89   6.18  4127586  0.032898        NaN
3 2018-07-05    ACB  6.04  6.18  5.69   5.77  4948503 -0.068646        NaN
4 2018-07-06    ACB  5.60  6.33  5.60   6.33  7854475  0.092628        NaN


### Triple-Barrier Labeling

In [6]:
def triple_barrier_for_symbol(
    g,
    pt_mult=2.0,
    sl_mult=2.0,
    horizon=10,
    ambiguous_policy="drop",   # "drop" | "open_based"
    vb_policy="zero",          # "zero" | "sign"
    neutral_width=0.25,        # in units of daily_vol
):
    n = len(g)

    labels = np.full(n, np.nan)
    hit_dates = [pd.NaT] * n
    hit_types = [None] * n
    vertical_dates = [pd.NaT] * n
    upper_barriers = np.full(n, np.nan)
    lower_barriers = np.full(n, np.nan)
    event_returns = np.full(n, np.nan)

    close_arr = g["close"].to_numpy()
    open_arr  = g["open"].to_numpy()
    high_arr  = g["high"].to_numpy()
    low_arr   = g["low"].to_numpy()
    vol_arr   = g["daily_vol"].to_numpy()
    date_arr  = g["date"].to_numpy()

    for i in range(n):
        vol = vol_arr[i]
        if np.isnan(vol):
            continue

        j_end = min(i + horizon, n - 1)
        if j_end <= i:
            continue

        entry = close_arr[i]
        upper = entry * np.exp(pt_mult * vol)
        lower = entry * np.exp(-sl_mult * vol)

        upper_barriers[i] = upper
        lower_barriers[i] = lower
        vertical_dates[i] = pd.Timestamp(date_arr[j_end])

        label = np.nan
        hit_idx = j_end
        hit_type = "vb"

        for j in range(i + 1, j_end + 1):
            up_hit = high_arr[j] >= upper
            dn_hit = low_arr[j] <= lower

            if up_hit and dn_hit:
                if ambiguous_policy == "drop":
                    label = np.nan
                    hit_idx = j
                    hit_type = "ambiguous"
                else:
                    # imperfect but deterministic fallback
                    d_up = abs(np.log(upper / open_arr[j]))
                    d_dn = abs(np.log(open_arr[j] / lower))
                    label = 1 if d_up < d_dn else -1
                    hit_idx = j
                    hit_type = "ambiguous_open_based"
                break

            elif up_hit:
                label = 1
                hit_idx = j
                hit_type = "pt"
                break

            elif dn_hit:
                label = -1
                hit_idx = j
                hit_type = "sl"
                break

        ret = np.log(close_arr[hit_idx] / entry)
        event_returns[i] = ret
        hit_dates[i] = pd.Timestamp(date_arr[hit_idx])
        hit_types[i] = hit_type

        if hit_type == "vb":
            if vb_policy == "zero":
                label = 0
            else:
                # make 0 a true "no move" zone, not a catch-all bucket
                if abs(ret) <= neutral_width * vol:
                    label = 0
                else:
                    label = 1 if ret > 0 else -1

        labels[i] = label

    out = g.copy()
    out["tb_label"] = labels
    out["tb_hit_date"] = hit_dates
    out["tb_hit_type"] = hit_types
    out["tb_vertical_date"] = vertical_dates
    out["tb_upper_barrier"] = upper_barriers
    out["tb_lower_barrier"] = lower_barriers
    out["tb_event_log_return"] = event_returns
    return out

In [7]:
# --------------------------------------------------
# Apply to all symbols
# --------------------------------------------------
labeled_list = []

# Triple-barrier parameters
VERTICAL_HORIZON = 10      # number of trading days forward
PT_MULT = 2.5              # profit-taking barrier = + PT_MULT * vol
SL_MULT = 2.5              # stop-loss barrier     = - SL_MULT * vol

for symbol, g in df.groupby("symbol", sort=True):
    labeled_g = triple_barrier_for_symbol(
        g,
        pt_mult=PT_MULT,
        sl_mult=SL_MULT,
        horizon=VERTICAL_HORIZON
    )
    labeled_list.append(labeled_g)

labeled = pd.concat(labeled_list, axis=0, ignore_index=True)
labeled = labeled.sort_values(["symbol", "date"]).reset_index(drop=True)

print(labeled["tb_label"].value_counts(dropna=False))

tb_label
 1.0    24637
-1.0    21553
 0.0    14549
 NaN      750
Name: count, dtype: int64


## 2. Feature Engineering

### Feature Construction

In [8]:
def make_features(
    df: pd.DataFrame,
    return_lags = (1, 3, 5, 10, 20),
    regime_windows = (10, 20),
    trend_window: int = 20,
    short_window: int = 5,
    long_window: int = 20,
    rsi_period: int = 14,
    include_calendar: bool = True,
    eps: float = 1e-12,
) -> pd.DataFrame:
    """
    Compact, scale-invariant feature builder.

    - standardized multi-horizon returns
    - normalized volatility regime
    - normalized volume surprise
    - one normalized distance-to-MA feature
    - one MA spread feature
    - one oscillator (RSI)
    - two normalized range/gap features
    - optional calendar features

    The explicit model feature list is stored in:
        out.attrs["feature_cols"]
    """

    required_cols = [
        "date", "symbol", "open", "high", "low", "close", "volume",
        "log_ret", "daily_vol"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if long_window < short_window:
        raise ValueError("long_window must be >= short_window")

    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    out = out.sort_values(["symbol", "date"]).reset_index(drop=True)
    g = out.groupby("symbol", group_keys=False)

    feature_cols = []
    helper_cols = []

    out["log_volume"] = np.log1p(out["volume"])
    helper_cols.append("log_volume")

    daily_vol_safe = out["daily_vol"].replace(0, np.nan)

    def _rolling_zscore(s: pd.Series, win: int) -> pd.Series:
        mu = s.rolling(win, min_periods=win).mean()
        sd = s.rolling(win, min_periods=win).std()
        return (s - mu) / (sd + eps)

    def _rsi(series: pd.Series, period: int = 14) -> pd.Series:
        delta = series.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        avg_gain = gain.rolling(period, min_periods=period).mean()
        avg_loss = loss.rolling(period, min_periods=period).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        return 100 - (100 / (1 + rs))

    # 1) Standardized multi-horizon returns
    for lag in sorted(set(return_lags)):
        temp_col = f"_log_ret_{lag}"
        ref_win = max(max(regime_windows), lag)

        out[temp_col] = g["close"].transform(
            lambda s, lag=lag: np.log(s / s.shift(lag))
        )
        helper_cols.append(temp_col)

        feat = f"z_log_ret_{lag}"
        out[feat] = out.groupby("symbol")[temp_col].transform(
            lambda s, win=ref_win: _rolling_zscore(s, win)
        )
        feature_cols.append(feat)

    # 2) Volatility regime: keep normalized volatility only
    for win in sorted(set(regime_windows)):
        temp_col = f"_volatility_{win}"
        out[temp_col] = g["log_ret"].transform(
            lambda s, win=win: s.rolling(win, min_periods=win).std()
        )
        helper_cols.append(temp_col)

        feat = f"z_volatility_{win}"
        out[feat] = out.groupby("symbol")[temp_col].transform(
            lambda s, win=win: _rolling_zscore(s, win)
        )
        feature_cols.append(feat)

    # 3) Volume surprise: keep z-scored log volume only
    for win in sorted(set(regime_windows)):
        log_vol_ma = g["log_volume"].transform(
            lambda s, win=win: s.rolling(win, min_periods=win).mean()
        )
        log_vol_sd = g["log_volume"].transform(
            lambda s, win=win: s.rolling(win, min_periods=win).std()
        )

        feat = f"z_log_volume_{win}"
        out[feat] = (out["log_volume"] - log_vol_ma) / (log_vol_sd + eps)
        feature_cols.append(feat)

    # 4) Trend features: one normalized dist-to-MA + one MA spread
    dist_col = f"_dist_ma_{trend_window}_pct"
    out[dist_col] = (
        out["close"]
        - g["close"].transform(lambda s: s.rolling(trend_window, min_periods=trend_window).mean())
    ) / (
        g["close"].transform(lambda s: s.rolling(trend_window, min_periods=trend_window).mean()) + eps
    )
    helper_cols.append(dist_col)

    feat = f"z_dist_ma_{trend_window}"
    out[feat] = out.groupby("symbol")[dist_col].transform(
        lambda s, win=trend_window: _rolling_zscore(s, win)
    )
    feature_cols.append(feat)

    ma_short = g["close"].transform(
        lambda s: s.rolling(short_window, min_periods=short_window).mean()
    )
    ma_long = g["close"].transform(
        lambda s: s.rolling(long_window, min_periods=long_window).mean()
    )

    feat = f"ma_spread_{short_window}_{long_window}"
    out[feat] = (ma_short / (ma_long + eps)) - 1.0
    feature_cols.append(feat)

    # 5) One oscillator only: centered RSI
    feat = f"rsi_{rsi_period}_centered"
    out[feat] = (g["close"].transform(lambda s: _rsi(s, rsi_period)) - 50.0) / 50.0
    feature_cols.append(feat)

    # 6) Compact normalized range/gap block
    prev_close = g["close"].shift(1)

    feat = "hl_range_vol_adj"
    out[feat] = (
        (out["high"] - out["low"]) / (out["close"] + eps)
    ) / (daily_vol_safe + eps)
    feature_cols.append(feat)

    feat = "open_to_prev_close_vol_adj"
    out[feat] = (
        out["open"] / (prev_close + eps) - 1.0
    ) / (daily_vol_safe + eps)
    feature_cols.append(feat)

    # 7) Calendar
    if include_calendar:
        out["day_of_week"] = out["date"].dt.dayofweek
        out["month"] = out["date"].dt.month
        feature_cols.extend(["day_of_week", "month"])

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.drop(columns=helper_cols, errors="ignore")

    # Use this exact list for modeling so raw columns are excluded.
    out.attrs["feature_cols"] = feature_cols
    return out

In [9]:
df_feat = make_features(labeled)
print(df_feat.columns.tolist())
print("Num of cols: ", len(df_feat.columns.tolist()))
print(df_feat.head(5))

['date', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'log_ret', 'daily_vol', 'tb_label', 'tb_hit_date', 'tb_hit_type', 'tb_vertical_date', 'tb_upper_barrier', 'tb_lower_barrier', 'tb_event_log_return', 'z_log_ret_1', 'z_log_ret_3', 'z_log_ret_5', 'z_log_ret_10', 'z_log_ret_20', 'z_volatility_10', 'z_volatility_20', 'z_log_volume_10', 'z_log_volume_20', 'z_dist_ma_20', 'ma_spread_5_20', 'rsi_14_centered', 'hl_range_vol_adj', 'open_to_prev_close_vol_adj', 'day_of_week', 'month']
Num of cols:  32
        date symbol  open  high   low  close   volume   log_ret  daily_vol  \
0 2018-07-02    ACB  6.87  6.87  6.33   6.43  6858520       NaN        NaN   
1 2018-07-03    ACB  6.45  6.56  5.91   5.98  7375018 -0.072554        NaN   
2 2018-07-04    ACB  5.89  6.18  5.89   6.18  4127586  0.032898        NaN   
3 2018-07-05    ACB  6.04  6.18  5.69   5.77  4948503 -0.068646        NaN   
4 2018-07-06    ACB  5.60  6.33  5.60   6.33  7854475  0.092628        NaN   

   tb_label  ... z_volat

### Feature Selection

In [10]:
exclude_cols = labeled.columns.tolist()

feature_cols = [c for c in df_feat.columns if c not in exclude_cols]
print("Final feature columns:", len(feature_cols))
print(feature_cols)

Final feature columns: 16
['z_log_ret_1', 'z_log_ret_3', 'z_log_ret_5', 'z_log_ret_10', 'z_log_ret_20', 'z_volatility_10', 'z_volatility_20', 'z_log_volume_10', 'z_log_volume_20', 'z_dist_ma_20', 'ma_spread_5_20', 'rsi_14_centered', 'hl_range_vol_adj', 'open_to_prev_close_vol_adj', 'day_of_week', 'month']


In [11]:
df_model = df_feat.dropna(subset=feature_cols + ["tb_label", "tb_hit_date"]).copy()
df_model.head()

,date,symbol,open,high,low,close,volume,log_ret,daily_vol,tb_label,...,z_volatility_20,z_log_volume_10,z_log_volume_20,z_dist_ma_20,ma_spread_5_20,rsi_14_centered,hl_range_vol_adj,open_to_prev_close_vol_adj,day_of_week,month
39,2018-08-24,ACB,7.37,7.59,7.32,7.49,7614923,0.016151,0.014127,-1.0,...,-1.166067,1.288939,1.660917,1.318672,0.030294,0.546875,2.551798,-9.603859e-12,4,8
40,2018-08-27,ACB,7.53,7.63,7.49,7.49,3180882,0.000000,0.013950,-1.0,...,-1.148843,-0.877111,-0.931470,1.148517,0.038046,0.524590,1.339942,3.828406e-01,0,8
41,2018-08-28,ACB,7.51,7.57,7.49,7.51,5158953,0.002667,0.013653,-1.0,...,-1.143024,0.374160,0.467528,1.006274,0.039005,0.425743,0.780218,1.955754e-01,1,8
42,2018-08-29,ACB,7.59,7.63,7.49,7.59,4615659,0.010596,0.013374,-1.0,...,-1.197420,0.135910,0.094878,1.226342,0.041725,0.457944,1.379175,7.964954e-01,2,8
43,2018-08-30,ACB,7.61,7.66,7.57,7.66,4212097,0.009180,0.013346,-1.0,...,-1.203823,-0.000517,-0.167827,1.297981,0.043911,0.442308,0.880340,1.974353e-01,3,8


In [12]:
# --------------------------------------------------
# Save labeled dataset
# --------------------------------------------------
df_model.to_csv(PROCESSED_OUT, index=False)
print("Saved:", PROCESSED_OUT)

Saved: ./data\processed_vn30_panel_daily.csv


## 3. Data Splitting

### Purged Time Series Cross-Validation

In [14]:
class PurgedTimeSeriesSplit:
    def __init__(
        self,
        n_splits=5,
        embargo_days=10,
        min_train_size=252
    ):
        self.n_splits = n_splits
        self.embargo_days = embargo_days
        self.min_train_size = min_train_size

    def split(self, df):
        df = df.sort_values("date").reset_index(drop=True).copy()
        unique_dates = np.array(sorted(df["date"].unique()))

        n_dates = len(unique_dates)
        fold_size = n_dates // (self.n_splits + 1)

        for fold in range(self.n_splits):
            val_start_idx = (fold + 1) * fold_size
            if fold == self.n_splits - 1:
                val_end_idx = n_dates - 1
            else:
                val_end_idx = (fold + 2) * fold_size - 1

            val_start_date = unique_dates[val_start_idx]
            val_end_date = unique_dates[val_end_idx]
            embargo_end_date = pd.Timestamp(val_end_date) + pd.Timedelta(days=self.embargo_days)

            val_mask = (df["date"] >= val_start_date) & (df["date"] <= val_end_date)

            train_mask = df["date"] < val_start_date

            # purge training rows whose event overlaps validation
            overlap_mask = train_mask & (df["tb_hit_date"] >= val_start_date)
            train_mask = train_mask & (~overlap_mask)

            # embargo after validation
            embargo_mask = (df["date"] > val_end_date) & (df["date"] <= embargo_end_date)
            train_mask = train_mask & (~embargo_mask)

            train_idx = np.where(train_mask)[0]
            val_idx = np.where(val_mask)[0]

            if len(train_idx) < self.min_train_size:
                continue
            if len(val_idx) == 0:
                continue

            yield train_idx, val_idx

### Time-series Train - Validation / Test Split

In [15]:
def chronological_holdout_split(df, test_start="2025-01-01"):
    trainval = df[df["date"] < pd.to_datetime(test_start)].copy()
    test = df[df["date"] >= pd.to_datetime(test_start)].copy()
    return trainval, test

In [16]:
trainval_df, test_df = chronological_holdout_split(df_model, test_start="2025-01-01")

In [17]:
print("trainval:", trainval_df["date"].min(), "->", trainval_df["date"].max(), len(trainval_df))
print("test    :", test_df["date"].min(), "->", test_df["date"].max(), len(test_df))

trainval: 2018-08-24 00:00:00 -> 2024-12-31 00:00:00 50756
test    : 2025-01-02 00:00:00 -> 2026-03-12 00:00:00 9357


In [18]:
def evaluate(
    clf,
    test_df,
    feature_cols,
):
    X_test = test_df[feature_cols]
    y_test = test_df["tb_label"]

    if clf.__class__.__name__ == "XGBClassifier":
        y_train_enc, y_test_enc, le = encode_labels(y_train, y_test)
        pred_enc = clf.predict(X_test)
        y_pred = le.inverse_transform(pred_enc.astype(int))
    else:
        y_pred = clf.predict(X_test)

    metrics = compute_metrics(y_test, y_pred)

    print("\nFinal test metrics:")
    print(metrics)

    print("\nClassification report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    labels_sorted = sorted(trainval_df["tb_label"].unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)

    print("\nConfusion matrix (labels order =", labels_sorted, "):")
    print(cm)

    return {
        "model": clf,
        "metrics": metrics,
        "y_test": y_test,
        "y_pred": y_pred,
        "confusion_matrix": cm,
    }

In [29]:
X_trainval = trainval_df[feature_cols]
y_train = trainval_df["tb_label"]   # IMPORTANT: create global y_train for your evaluate(...)

In [ ]:


from stacking_model import TimeSeriesStackingClassifier

stacker = TimeSeriesStackingClassifier(
    feature_cols=feature_cols,
    cv_splits=5,
    embargo_days=10,
    min_train_size=252 * 2,
    meta_C=1.0,
)

stacker.fit(trainval_df, cv_results_path="results/cross_val.csv")
stack_res = stacker.evaluate(test_df)
stacker.save("./models")